# Genotype–Phenotype Heterogeneity and Heterogeneity-Driven Infertility Management in Non-Classical 21-Hydroxylase Deficiency: A Clinical Analysis and Literature Review Exploration with `mlcroissant`

This notebook demonstrates how to load and analyze the FAIR^2 clinical dataset using the `mlcroissant` library. All data entities are referenced by their `@id`s to ensure reproducibility and clarity.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.cbsv-djdb/fair2.json`

The dataset includes structured records on clinical features, hormone levels, adrenal imaging, and genetic variants from three female patients with non-classical 21-hydroxylase deficiency-associated infertility.


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.cbsv-djdb/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

# Display metadata name and description
print(f"Dataset title: {dataset.metadata.name}")
print(f"Description: {dataset.metadata.description}")
print(f"Authors (by @id): {dataset.metadata.author}")
print(f"RecordSets (by @id): {[rs['@id'] for rs in getattr(dataset.metadata, 'recordSet', [])]}")

## 2. Data Overview
Review available record sets, fields, and their IDs. In this step, we enumerate the dataset's record sets and sample the first few entities for each.

In [ ]:
# List all available record sets (by @id)
record_sets = [rs['@id'] for rs in getattr(dataset.metadata, 'recordSet', [])]
print("Available RecordSets @id:")
for rs_id in record_sets:
    print("  -", rs_id)

# Show sample records for each RecordSet using their @id
for record_set_id in record_sets:
    print(f"\nSample records for RecordSet @id: {record_set_id}")
    for i, record in enumerate(dataset.records(record_set=record_set_id)):
        print(record)
        if i >= 2:
            print("...")
            break


## 3. Data Extraction
Load data from each record set into a DataFrame for analysis. All entity-level references use their `@id`. The available record sets typically represent:
- Table 1: Clinical features at diagnosis
- Table 2: Hormone levels and imaging
- Table 3: Genetic variant annotations

This code dynamically loads all record sets referenced by their `@id`.

In [ ]:
# Extract data from each record set (@id-driven)
dataframes = {}
for record_set_id in record_sets:
    records = list(dataset.records(record_set=record_set_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df

# Preview columns and records for the primary clinical table
main_recordset_id = record_sets[0] if record_sets else None
if main_recordset_id and main_recordset_id in dataframes:
    print(f"Columns in RecordSet @id {main_recordset_id}:")
    print(dataframes[main_recordset_id].columns.tolist())
    print(dataframes[main_recordset_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps — filtering records by a numeric field `@id`, normalizing data, and grouping by another field. All references are via the dataset's schema `@id`.

Let's suppose Table 1 includes an age field with @id `'age_at_diagnosis'`, and Table 2 includes a hormone measurement with @id `'serum_17OHP'`. We demonstrate outlier removal, normalization, and grouping.


In [ ]:
# Choose a RecordSet and numeric field for analysis
clinical_rs = main_recordset_id
clinical_df = dataframes.get(clinical_rs, pd.DataFrame())

# Suppose 'age_at_diagnosis' is a numeric field @id
numeric_field_id = 'age_at_diagnosis'  # Example field @id, change if actual differs

if numeric_field_id in clinical_df.columns:
    threshold = 15
    filtered_df = clinical_df[clinical_df[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold}:")
    print(filtered_df.head())
    
    # Normalize the numeric field
    filtered_df[f"{numeric_field_id}_normalized"] = (
        filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
    ) / filtered_df[numeric_field_id].std()
    print(f"Normalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())
    
    # Group samples by diagnosis status (suppose field @id 'oligomenorrhea')
    group_field_id = 'oligomenorrhea'
    if group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean()
        print(f"Grouped mean {numeric_field_id} by {group_field_id}:")
        print(grouped_df.head())
else:
    print(f"Field @id '{numeric_field_id}' not found in clinical_df columns: {clinical_df.columns}")

## 5. Visualization
Visualize data distributions or relationships using fields referenced by their `@id`.

Here we plot the distribution of age at diagnosis and illustrate the relationship between age and another clinical field (e.g., height).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plotting age distribution and its relationship to height
if numeric_field_id in clinical_df.columns:
    plt.figure(figsize=(6, 4))
    sns.histplot(clinical_df[numeric_field_id], bins=5, kde=True)
    plt.title('Distribution of Age at Diagnosis (@id: age_at_diagnosis)')
    plt.xlabel('Age at Diagnosis')
    plt.ylabel('Count')
    plt.show()

    # Scatter plot age vs height
    height_field_id = 'height'  # Assumed @id for height
    if height_field_id in clinical_df.columns:
        plt.figure(figsize=(6, 4))
        sns.scatterplot(x=clinical_df[numeric_field_id], y=clinical_df[height_field_id])
        plt.title('Age at Diagnosis vs Height')
        plt.xlabel('Age at Diagnosis')
        plt.ylabel('Height')
        plt.show()
else:
    print(f"Field @id '{numeric_field_id}' not found for plotting.")

## 6. Conclusion
In this notebook, we've loaded and explored the FAIR^2 clinical dataset using `mlcroissant` with strict referencing by `@id` throughout.

- All data entities (record sets, fields, columns) are accessed consistently via their `@id`.
- We previewed clinical, hormonal, and genetic variant tables, performed basic EDA, and visualized distributions.

This workflow can be reused for any Croissant-encoded dataset, supporting reproducible biomedical analytics and interoperable FAIR data sharing.
